<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/final_thesis_with_RAG_%26_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install langchain openai faiss-cpu pypdf docx2txt langchain-openai langchain-community

In [23]:
!pip install gradio

### Gradio Web Interface for QA System

After executing the cell above, a public URL will appear in the output. You can use this URL to access your Gradio application and share it with your professor. Please note that the Gradio app will keep running in the background until you stop the Colab runtime or interrupt the cell execution.

In [24]:
import os
from pathlib import Path

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore
from langchain_core.prompts import ChatPromptTemplate

In [25]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

CHUNK_SIZE       = 500       # characters per chunk
CHUNK_OVERLAP    = 50        # overlap between consecutive chunks
TOP_K            = 4         # number of chunks to retrieve per query
LLM_MODEL        = "gpt-3.5-turbo"   # older ChatGPT model

### Extract entities + relations via LLM

In [26]:
import json
import requests
import os
from openai import OpenAI
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


# ─── Step 0: Extract entities + relations + intent via OpenAI ─────────────────

def extract_components(question: str) -> dict:
    prompt = f"""
You are an NLP assistant for Wikidata SPARQL query generation.
Given a natural language question, extract the following as JSON:

- entities: list of named entities (people, places, things)
- relations: list of relations/properties being asked about
- intent: one of SELECT, ASK, COUNT
- filters: any constraints (dates, numbers, etc.)
- answer_type: what kind of value is expected (e.g. place, date, person, number)

Question: "{question}"

Respond ONLY with a valid JSON object, no explanation.
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_response_content = response.choices[0].message.content
    # Remove markdown code block delimiters if present
    if raw_response_content.startswith('```json') and raw_response_content.endswith('```'):
        raw_response_content = raw_response_content[len('```json'):-len('```')].strip()
    try:
        return json.loads(raw_response_content)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI. Raw response: '{raw_response_content}'")
        raise e

### Evaluate and improve user's question

In [27]:
def improve_question_with_llm(question: str) -> str:
    prompt = f"""
    You are an expert at improving natural language questions for search engines.
    Given a question, correct any spelling or grammatical errors, and rephrase it to be clearer and more concise for a factual search, if necessary.
    If the question is already clear and correct, return it as is.

    Example 1:
    Question: "who is the current president of Alban?"
    Improved Question: "who is the current president of Albania?"

    Example 2:
    Question: "tell me about capital of france"
    Improved Question: "What is the capital of France?"

    Example 3:
    Question: "what is the population of the city with the eiffel tower"
    Improved Question: "What is the population of Paris?"

    Question: "{question}"
    Improved Question:
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

### LIniking entities via Falcon 2.0

In [28]:
# ─── Step 1: Link entities to Wikidata Q/P numbers via Falcon 2.0 ─────────────

def link_to_wikidata(question: str) -> dict:
    url = "https://labs.tib.eu/falcon/falcon2/api?mode=short"
    headers = {"Content-Type": "application/json"}
    payload = {"text": question}
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()

    entities = [
        {"label": e.get("label"), "uri": e.get("uri")}
        for e in data.get("entities_wikidata", [])
    ]
    relations = [
        {"label": r.get("label"), "uri": r.get("uri")}
        for r in data.get("relations_wikidata", [])
    ]
    return {"entities": entities, "relations": relations}

### LInking entities via Wikidata API

In [29]:
# ─── Step 2: Linking entities to Wikidata via Wikidata API ─────────────────

# import requests

# def link_entity_to_qid(entity_name: str, language: str = "en") -> list:
#     """
#     Use the Wikidata search API to find QID candidates for an entity name.
#     Returns a ranked list of candidates with QID, label, and description.
#     """
#     url = "https://www.wikidata.org/w/api.php"
#     params = {
#         "action": "wbsearchentities",
#         "search": entity_name,
#         "language": language,
#         "format": "json",
#         "limit": 5
#     }
#     # Add a User-Agent header
#     headers = {
#         "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
#     }
#     response = requests.get(url, params=params, headers=headers)

#     # Always print status code and response content for debugging
#     print(f"Wikidata API Response Status Code: {response.status_code}")
#     print(f"Wikidata API Response Content: {response.text}")

#     # Raise an HTTPError for bad responses (4xx or 5xx)
#     response.raise_for_status()

#     try:
#         if not response.text.strip(): # Check if the response content is empty or only whitespace
#             print("Wikidata API returned an empty or whitespace-only response.")
#             return [] # Return empty list if no content to parse
#         data = response.json()
#     except requests.exceptions.JSONDecodeError as e:
#         print(f"JSONDecodeError from Wikidata API after status check. Raw response text: '{response.text}'")
#         raise e

#     candidates = []
#     for result in data.get("search", []):
#         candidates.append({
#             "qid": result["id"],
#             "label": result.get("label", ""),
#             "description": result.get("description", ""),
#         })
#     return candidates
# print(link_entity_to_qid("simulation"))

### Generate SPARQL queries via LLM

In [30]:
# ─── Step 3: Generate SPARQL query via OpenAI ─────────────────────────────────

def generate_sparql(question: str, components: dict, linked: dict) -> str:
    prompt = f"""
You are a SPARQL expert for Wikidata.
Generate a valid Wikidata SPARQL query for the following question.

Question: "{question}"

Extracted components:
{json.dumps(components, indent=2)}

Wikidata linked entities and relations:
{json.dumps(linked, indent=2)}

Rules:
- Use the Wikidata SPARQL endpoint format (wd:, wdt:, p:, ps:, pq:)
- Use SERVICE wikibase:label for labels
- Use LIMIT 10 unless a specific count is requested
- NEVER use property paths with * or + (e.g. wdt:P171*) — they time out
- NEVER use deeply nested subqueries
- Keep queries flat: maximum 3-4 triple patterns per query
- For multi-hop questions, generate only the FIRST hop query
  and note which variable needs a second query
- For time-sensitive properties (current officeholder, current population),
  use the qualifier pattern:
    p:PXX ?stmt . ?stmt ps:PXX ?value .
    FILTER NOT EXISTS {{ ?stmt pq:P582 ?end }}
- Return ONLY the SPARQL query, no explanation

SPARQL:
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_sparql_query = response.choices[0].message.content.strip()
    # Remove markdown code block delimiters if present
    if raw_sparql_query.startswith('```sparql') and raw_sparql_query.endswith('```'):
        raw_sparql_query = raw_sparql_query[len('```sparql'):-len('```')].strip()
    return raw_sparql_query

### Query Wikidata

In [31]:
# ─── Step 4: Run the SPARQL query against Wikidata ────────────────────────────

def run_sparql(query: str) -> list:
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    print(f"Raw SPARQL response status: {response.status_code}")
    print(f"Raw SPARQL response text: {response.text}")
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return data["results"]["bindings"]


### Main pipeline

In [32]:
# ─── Main pipeline ────────────────────────────────────────────────────────────

def answer_question(question: str):
    print(f"\nQuestion: {question}\n")

    print("Step 1: Extracting components...")
    components = extract_components(question)
    print(json.dumps(components, indent=2))

    print("\nStep 2: Linking to Wikidata...")
    linked = link_to_wikidata(question)
    print(json.dumps(linked, indent=2))

    print("\nStep 3: Generating SPARQL...")
    sparql = generate_sparql(question, components, linked)
    print(sparql)

    print("\nStep 4: Querying Wikidata...")
    results = run_sparql(sparql)
    print(f"Got {len(results)} result(s):")
    for r in results:
        for key, val in r.items():
            print(f"  {key}: {val['value']}")

    return results

# User's question here:

In [33]:
# Define the question once here
user_question = "how many ingredients does creatine contain?"
user_question_llm = improve_question_with_llm(user_question)
print(f"Original Question: {user_question}")
print(f"Improved Question (LLM): {user_question_llm}")

Original Question: how many ingredients does creatine contain?
Improved Question (LLM): "How many ingredients are in creatine?"


### Getting Wikidata's answer

In [34]:
print(f"\nRunning Wikidata pipeline for question: {user_question_llm}\n")
wikidata_raw_results = answer_question(user_question_llm)


Running Wikidata pipeline for question: "How many ingredients are in creatine?"


Question: "How many ingredients are in creatine?"

Step 1: Extracting components...
{
  "entities": [
    "creatine"
  ],
  "relations": [
    "ingredients"
  ],
  "intent": "COUNT",
  "filters": [],
  "answer_type": "number"
}

Step 2: Linking to Wikidata...
{
  "entities": [
    {
      "label": null,
      "uri": null
    }
  ],
  "relations": []
}

Step 3: Generating SPARQL...
SELECT (COUNT(?ingredient) AS ?ingredientCount) WHERE {
  wd:Q407885 wdt:P527 ?ingredient.
}

Step 4: Querying Wikidata...
Raw SPARQL response status: 200
Raw SPARQL response text: {
  "head" : {
    "vars" : [ "ingredientCount" ]
  },
  "results" : {
    "bindings" : [ {
      "ingredientCount" : {
        "datatype" : "http://www.w3.org/2001/XMLSchema#integer",
        "type" : "literal",
        "value" : "0"
      }
    } ]
  }
}
Got 1 result(s):
  ingredientCount: 0


### Instructions -prompting LLM about its answers

In [35]:
def ask_llm_directly(question: str) -> str:
    prompt = f"""
    Answer the following question without repeating user's question in your answer.

    Question: "{question}"

    Answer:
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

### Getting LLM's answer

In [36]:
if __name__ == "__main__":
    question_for_llm = user_question_llm
    print(f"\nQuestion for LLM: {question_for_llm}\n")
    llm_direct_answer = ask_llm_directly(question_for_llm)
    print(f"LLM's Answer: {llm_direct_answer}")


Question for LLM: "How many ingredients are in creatine?"

LLM's Answer: Creatine itself is a single compound, so it consists of one main ingredient. However, if you are referring to a creatine supplement, it may contain additional ingredients such as flavorings, sweeteners, or other additives, depending on the product.


### RAG PIPELINE

In [37]:
# ─────────────────────────────────────────────
# STEP 1 — UPLOAD / LOAD FILE
# ─────────────────────────────────────────────

def load_file(file_path: str) -> list[Document]:
    """
    Load a document from disk.
    Supports: .pdf, .txt, .docx
    Returns a list of LangChain Document objects.
    """
    path = Path(file_path)
    suffix = path.suffix.lower()

    print(f"[1/4] Loading file: {path.name}  (type: {suffix})")

    if suffix == ".pdf":
        loader = PyPDFLoader(file_path)
    elif suffix == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    elif suffix == ".docx":
        loader = Docx2txtLoader(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}. Use .pdf, .txt, or .docx")

    documents = loader.load()
    print(f"    Loaded {len(documents)} page(s) / section(s)")
    return documents

In [38]:
# ─────────────────────────────────────────────
# STEP 2 — CHUNKING
# ─────────────────────────────────────────────

def chunk_documents(documents: list[Document]) -> list[Document]:
    """
    Split documents into smaller overlapping chunks.
    Uses RecursiveCharacterTextSplitter — tries to split on
    paragraphs, then sentences, then words, then characters.
    """
    print(f"[2/4] Chunking  (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(documents)
    print(f"    Produced {len(chunks)} chunk(s)")
    return chunks

In [39]:
# ─────────────────────────────────────────────
# STEP 3 — EMBED + STORE IN FAISS
# ─────────────────────────────────────────────
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

def embed_and_store(chunks: list[Document]) -> VectorStore:
    """
    Convert chunks to embeddings using OpenAI text-embedding-ada-002
    and store them in FAISS.
    Returns a LangChain FAISS vectorstore object.
    """
    print(f"[3/4] Embedding {len(chunks)} chunk(s) → storing in FAISS")


    # embeddings model
    embeddings = OpenAIEmbeddings(
        model="text-embedding-ada-002",
    )

    # store chunks in FAISS
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings,
    )

    print(f"    Stored in FAISS vectorstore.")
    return vectorstore

In [40]:
# ─────────────────────────────────────────────
# STEP 4 — RETRIEVAL + QA
# ─────────────────────────────────────────────

def build_qa_chain(vectorstore: VectorStore, system_message: str = None) -> RetrievalQA:
    """
    Build a RetrievalQA chain:
      - retriever: FAISS similarity search (top-k chunks)
      - LLM: gpt-3.5-turbo (older ChatGPT model)
      - chain type: stuff (concatenate chunks into one prompt)
    """
    print(f"[4/4] Building QA chain  (model={LLM_MODEL}, top_k={TOP_K})")

    llm = ChatOpenAI(
        openai_api_key=os.environ.get('OPENAI_API_KEY'),
        model_name=LLM_MODEL,
        temperature=0,
    )

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )

    # Define the prompt template with a system message if provided
    if system_message:
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", system_message),
            ("human", "Context: {context}\nQuestion: {question}")
        ])
        chain_kwargs = {"prompt": prompt_template}
    else:
        chain_kwargs = {}

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs=chain_kwargs
    )

    return qa_chain


def ask(qa_chain: RetrievalQA, question: str) -> dict:
    """
    Ask a question against the loaded document.
    Returns the answer and the source chunks used.
    """
    print(f"\nQuestion: {question}")
    result = qa_chain({"query": question})

    answer = result["result"]
    sources = result["source_documents"]

    print(f"Answer:   {answer}")
    print(f"Sources:  {len(sources)} chunk(s) retrieved")
    for i, doc in enumerate(sources, 1):
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  [{i}] ...{preview}...")

    return {"answer": answer, "sources": sources}

In [41]:
# ─────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(file_path: str, questions: list[str], system_message: str = None) -> str:
    """
    End-to-end pipeline:
      1. Load file
      2. Chunk
      3. Embed + store in FAISS
      4. Build QA chain
      5. Answer each question
    """
    documents   = load_file(file_path)
    chunks      = chunk_documents(documents)
    vectorstore = embed_and_store(chunks)
    qa_chain    = build_qa_chain(vectorstore, system_message=system_message)

    print("\n" + "─" * 50)
    # Assuming only one question is passed for this use case
    if questions:
        question = questions[0]
        result = ask(qa_chain, question)
        print("─" * 50)
        return result["answer"]
    return "No question provided."


In [42]:
# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    # Example of a system message to guide the LLM's answer style
    my_system_message = """You are a helpful assistant for a QA system.
    Provide concise answers and focus only on information found in the document.
    Do not hallucinate or add outside knowledge.
    If you don't know the answer tell the user that the information is not provided in the document.
    Do not repeat the question when answering. Just give the answer."""

    rag_pipeline_answer = run_pipeline(
        file_path="/content/MiCA1.docx",   # swap with your file path
        questions=[
            user_question_llm
        ],
        system_message=my_system_message # Pass the system message here
    )


[1/4] Loading file: MiCA1.docx  (type: .docx)
    Loaded 1 page(s) / section(s)
[2/4] Chunking  (size=500, overlap=50)
    Produced 52 chunk(s)
[3/4] Embedding 52 chunk(s) → storing in FAISS
    Stored in FAISS vectorstore.
[4/4] Building QA chain  (model=gpt-3.5-turbo, top_k=4)

──────────────────────────────────────────────────

Question: "How many ingredients are in creatine?"


/tmp/ipykernel_6253/3973674663.py:52: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": question})


Answer:   The information is not provided in the document.
Sources:  4 chunk(s) retrieved
  [1] .... Under MiCA, non-fungible tokens issued in large series could be considered fungible and therefore require an authoriza...
  [2] ...As use of blockchain technologies, digital assets, and cryptocurrencies has steadily risen in the past decade, regulator...
  [3] .... Learn more about progressive decentralization, how to apply 'decentralization test' to your project and how does the g...
  [4] ...By June 30, 2024, MiCA’s rules on issuing asset-referenced tokens (ARTs) and e-money tokens (EMTs) were enforced, and al...
──────────────────────────────────────────────────


### LLM-as-a-Judge Evaluation

In [43]:
def llm_judge(question: str, wikidata_result: list, llm_answer: str, rag_answer: str) -> dict:
    # Format Wikidata results for the LLM prompt
    formatted_wikidata_answers = []
    if wikidata_result:
        for binding in wikidata_result:
            found_label = False
            # First, try to find any key ending with 'Label'
            for key, value_dict in binding.items():
                if key.endswith('Label') and 'value' in value_dict:
                    formatted_wikidata_answers.append(value_dict['value'])
                    found_label = True
                    break

            # If no label was found for this binding, fallback to the first 'value' found in any key
            if not found_label:
                for key, value_dict in binding.items():
                    if 'value' in value_dict:
                        formatted_wikidata_answers.append(value_dict['value'])
                        break

    # If no answers were extracted, set a default message
    formatted_wikidata_answer_str = ", ".join(formatted_wikidata_answers) if formatted_wikidata_answers else "No answer found from Wikidata."

    prompt = f"""
    You are an expert evaluator for factual questions. Your task is to act as an LLM-as-a-judge.
    Given an original question and three answers (one obtained from Wikidata, one from a direct LLM call, and one from a RAG pipeline), perform the following:

    1. Judge each answer independently first. Do NOT let one answer influence scoring of the other.
    2. Evaluate the factual accuracy of each answer (Wikidata, direct LLM, and RAG) on a scale of 0-100%.
    2. Provide a brief explanation for each accuracy score, noting any specifics about the answer (e.g., if it's too brief, too verbose, or incorrect).
    3. If both answers are highly accurate (e.g., > 80%), provide a combined, concise answer to the original question. This combined answer should be more than one word, but not overly verbose, and synthesize the information.
    4. If one answer is significantly more accurate, favor that one for the combined answer.
    5. Do not assume agreement implies correctness. Both answers may be wrong.
    6. If neither is accurate, state that a combined answer is not possible.
    7. Penalize hallucinations and unsupported specifics.

    --------------------------------------------------
    SCORING RUBRIC (0-100)
    --------------------------------------------------

    Score each answer using:

    1. Factual Correctness (0-45)
    - Is it true?
    - Any fabricated claims?
    - Any contradictions?

    2. Completeness (0-35)
    - Does it fully answer the question?
    - Does the answer contain enough essential information to satisfy the question?
      For structured Wikidata answers, short labels or official names may count as complete if they directly answer the query.

    3. Relevance / Clarity (0-20)
    - Directly answers question?
    - Clear and concise?
    - Is the answer relevant and understandable?
      For short Wikidata answers, clarity means recognizable and unambiguous wording, not prose quality.

    Total = sum (0-100)

    --------------------------------------------------
    FINAL ANSWER RULES
    --------------------------------------------------

    If one answer scores at least 15 points higher than the other, prefer that answer.

    If all answers score >= 80 and are compatible, synthesize a concise better answer.

    If one answer is correct but incomplete, and the other adds correct useful details, merge them carefully.

    If all answers are weak (<60), set combined_answer = null.

    If uncertainty remains, say so briefly.

    Never copy false claims into the combined answer.

    --------------------------------------------------
    OUTPUT FORMAT
    --------------------------------------------------

    Return ONLY valid JSON.


    {{
      "user_question": "...",

      "wikidata_explanation": "...",
      "wikidata_answer_provided": "...",
      "wikidata_accuracy_percent": 0,
      "wikidata_breakdown": {{
        "factual_correctness": 0,
        "completeness": 0,
        "relevance_clarity": 0
      }},

      "llm_explanation": "...",
      "llm_direct_answer_provided": "...",
      "llm_accuracy_percent": 0,
      "llm_breakdown": {{
        "factual_correctness": 0,
        "completeness": 0,
        "relevance_clarity": 0
      }},

      "rag_explanation": "...",
      "rag_answer_provided": "...",
      "rag_accuracy_percent": 0,
      "rag_breakdown": {{
        "factual_correctness": 0,
        "completeness": 0,
        "relevance_clarity": 0
      }},


      "winner": "wikidata | llm | rag | merged | none",

      "confidence": 0.0,

      "combined_answer": "...",

      "needs_human_review": false
    }}

    Original Question: "{question}"
    Wikidata Answer: "{formatted_wikidata_answer_str}"
    Direct LLM Answer: "{llm_answer}"
    RAG Answer: "{rag_answer}"

    Respond ONLY with a valid JSON object, no explanation.
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_response_content = response.choices[0].message.content
    # Remove markdown code block delimiters if present
    if raw_response_content.startswith('```json') and raw_response_content.endswith('```'):
        raw_response_content = raw_response_content[len('```json'):-len('```')].strip()
    try:
        return json.loads(raw_response_content)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from LLM Judge. Raw response: '{raw_response_content}'")
        raise e

print(f"\n\n--- LLM-as-a-Judge Evaluation ---\n")
# Assuming 'rag_pipeline_answer' is available in the global scope
judge_results = llm_judge(user_question_llm, wikidata_raw_results, llm_direct_answer, rag_pipeline_answer)
print(json.dumps(judge_results, indent=2))




--- LLM-as-a-Judge Evaluation ---

{
  "user_question": "How many ingredients are in creatine?",
  "wikidata_explanation": "The answer '0' is incorrect as creatine is a compound and thus has one main ingredient.",
  "wikidata_answer_provided": "0",
  "wikidata_accuracy_percent": 20,
  "wikidata_breakdown": {
    "factual_correctness": 10,
    "completeness": 5,
    "relevance_clarity": 5
  },
  "llm_explanation": "The answer correctly identifies creatine as a single compound, but also notes that supplements may contain additional ingredients, which is accurate.",
  "llm_direct_answer_provided": "Creatine itself is a single compound, so it consists of one main ingredient. However, if you are referring to a creatine supplement, it may contain additional ingredients such as flavorings, sweeteners, or other additives, depending on the product.",
  "llm_accuracy_percent": 90,
  "llm_breakdown": {
    "factual_correctness": 45,
    "completeness": 30,
    "relevance_clarity": 15
  },
  "ra

In [ ]:
import gradio as gr
import json
import os
from google.colab import userdata

# Ensure OpenAI API key is set for Streamlit environment and other functions
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

def qa_interface(user_input: str, rag_document_file) -> tuple:
    if not user_input:
        return "Please enter a question.", "", "", "", "", ""

    # 1. Improve Question with LLM
    user_question_llm = improve_question_with_llm(user_input)

    # 2. Get Wikidata Answer
    wikidata_raw_results = answer_question(user_question_llm)

    formatted_wikidata_answers = []
    if wikidata_raw_results:
        for binding in wikidata_raw_results:
            found_label = False
            for key, value_dict in binding.items():
                if key.endswith('Label') and 'value' in value_dict:
                    formatted_wikidata_answers.append(value_dict['value'])
                    found_label = True
                    break
            if not found_label:
                for key, value_dict in binding.items():
                    if 'value' in value_dict:
                        formatted_wikidata_answers.append(value_dict['value'])
                        break
    wikidata_answer_str = ", ".join(formatted_wikidata_answers) if formatted_wikidata_answers else "No answer found from Wikidata."

    # 3. Get Direct LLM Answer
    llm_direct_answer = ask_llm_directly(user_question_llm)

    # 4. Get RAG Pipeline Answer
    rag_pipeline_answer = "N/A (No document uploaded or path provided)"
    if rag_document_file:
        # Save the uploaded file temporarily
        file_path = rag_document_file.name
        # Define system message for RAG pipeline
        my_system_message = """You are a helpful assistant for a QA system.\n    Provide concise answers and focus only on information found in the document.\n    Do not hallucinate or add outside knowledge.\n    If you don't know the answer tell the user that the information is not provided in the document.\n    Do not repeat the question when answering. Just give the answer."""
        rag_pipeline_answer = run_pipeline(
            file_path=file_path,
            questions=[
                user_question_llm
            ],
            system_message=my_system_message
        )

    # 5. LLM-as-a-Judge Evaluation
    judge_results = llm_judge(user_question_llm, wikidata_raw_results, llm_direct_answer, rag_pipeline_answer)

    # Format judge results for display
    judge_results_str = json.dumps(judge_results, indent=2)

    return (
        user_question_llm,
        wikidata_answer_str,
        llm_direct_answer,
        rag_pipeline_answer,
        judge_results_str
    )


# Gradio Interface
iface = gr.Interface(
    fn=qa_interface,
    inputs=[
        gr.Textbox(label="Enter your question here:", placeholder="e.g., How many ingredients are in creatine?"),
        gr.File(label="Upload document for RAG (optional, .pdf, .txt, .docx)")
    ],
    outputs=[
        gr.Textbox(label="Improved Question (LLM)"),
        gr.Textbox(label="Wikidata Answer"),
        gr.Textbox(label="Direct LLM Answer"),
        gr.Textbox(label="RAG Pipeline Answer"),
        gr.JSON(label="LLM-as-a-Judge Evaluation")
    ],
    title="Question Answering System Demo",
    description="Ask a question and see answers from Wikidata, a direct LLM call, and a RAG pipeline (if a document is provided), along with an LLM-as-a-Judge evaluation."
)

iface.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://10c00ef86c1ea4e3dc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Question: "What is mica?"

Step 1: Extracting components...
{
  "entities": [
    "mica"
  ],
  "relations": [],
  "intent": "SELECT",
  "filters": [],
  "answer_type": "thing"
}

Step 2: Linking to Wikidata...
{
  "entities": [
    {
      "label": null,
      "uri": null
    }
  ],
  "relations": []
}

Step 3: Generating SPARQL...
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q11423. # P31 (instance of) Q11423 (mica)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
LIMIT 10

Step 4: Querying Wikidata...
Raw SPARQL response status: 200
Raw SPARQL response text: {
  "head" : {
    "vars" : [ "item", "itemLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q968740"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "lunar mass"
      }
    }, {
      "item" : {
        "type" : "uri",
        "value" : 